# L2: Create Agents to Research and Write an Article (Modernized for CrewAI 1.x + Groq)

This notebook uses **Groq's Free Cloud API** via its native OpenAI-compatible endpoint.

Key highlights:
- **100% Free** & requires **0 GB disk space**.
- Ultra-fast cloud inference.
- Uses CrewAI's built-in native provider without requiring extra packages.

In [6]:
import os
import warnings
from pathlib import Path
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, LLM

warnings.filterwarnings('ignore')

# 1. Load Groq API key directly from .env
load_dotenv(override=True)
groq_api_key = os.getenv("GROQ_API_KEY", "").strip().strip('"\'')

if not groq_api_key or groq_api_key.startswith("your-"):
    raise ValueError(
        "GROQ_API_KEY is not set in .env! Please add your key to the .env file: GROQ_API_KEY=gsk_..."
    )

# 2. Connect to Groq's high-speed cloud endpoint
llm = LLM(
    model="openai/openai/gpt-oss-20b",
    base_url="https://api.groq.com/openai/v1",
    api_key=groq_api_key,
    temperature=0.7
)
print("✓ Connected to Groq Cloud LLM successfully!")

✓ Connected to Groq Cloud LLM successfully!


## Creating Agents

Define your Agents: `planner`, `writer`, and `editor`. Pass `llm=llm` directly.

In [7]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}. "
              "You collect key information to help the reader learn something valuable. "
              "Keep outlines focused and actionable.",
    llm=llm,
    verbose=True
)

writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're writing a concise article about the topic: {topic}. "
              "You base your writing on the Content Planner's outline. "
              "You provide objective and impartial insights clearly and concisely.",
    llm=llm,
    verbose=True
)

editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization.",
    backstory="You are an editor reviewing the blog post to ensure clarity, "
              "journalistic best practices, and balanced viewpoints.",
    llm=llm,
    verbose=True
)

## Creating Tasks

Define your sequential Tasks: `plan`, `write`, and `edit`.

In [8]:
plan = Task(
    description=(
        "1. Identify the key trends and takeaways on {topic}.\n"
        "2. Create a concise content outline (intro, 2 main points, conclusion).\n"
        "3. Keep the outline under 200 words."
    ),
    expected_output="A concise content outline under 200 words.",
    agent=planner,
)

write = Task(
    description=(
        "1. Use the content plan to craft a compelling, concise blog post on {topic}.\n"
        "2. Structure with an introduction, 2 short body sections, and a conclusion.\n"
        "3. Total length should be around 300-400 words."
    ),
    expected_output="A polished 300-400 word blog post in markdown format.",
    agent=writer,
)

edit = Task(
    description="Proofread and polish the given blog post for grammar, tone, and readability. Keep it concise.",
    expected_output="The final publication-ready markdown blog post.",
    agent=editor,
)

## Creating and Running the Crew

- `verbose=True` enables real-time execution logs.
- Tasks are executed sequentially by passing them in order.

In [9]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=True
)

result = crew.kickoff(inputs={"topic": "Jezreal Oseiwe Momoh"})

# In Modern CrewAI, result is a CrewOutput object with a .raw property
from IPython.display import Markdown
Markdown(result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 1fc123c8-f6a3-42b4-8c66-da7ab65a375b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 1. Identify the key trends and takeaways on Jezreal Oseiwe Momoh.                                        │
│  2. Create a concise content outline (intro, 2 main points, conclusion).                                        │
│  3. Keep the outline under 200 words.                                                                           │
│  ID: f1b707cc-4803-4ea7-9458-5428e89ca6cc                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: 1. Identify the key trends and takeaways on Jezreal Oseiwe Momoh.                                        │
│  2. Create a concise content outline (intro, 2 main points, conclusion).                                        │
│  3. Keep the outline under 200 words.                                                                           │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: 1fc123c8-f6a3-42b4-8c66-da7ab65a375b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

RuntimeError: Agent execution was invoked synchronously from within a running event loop. Use `agent.kickoff_async()` / `crew.kickoff_async()` (or `await agent.aexecute_task(...)`) when calling from async code.

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
# Inspect token usage metrics
print("Execution Token Usage:")
print(result.token_usage)